# 98 — Build combined nodal + Geode supergathers

Creates a provenance-rich supergather index for common Geode/nodal shots. It attempts to read Geode files with ObsPy where possible, but is designed to remain useful even when Geode conversion needs later format-specific work.

Outputs/replaces:
- `combined_supergathers`
- `combined_supergather_files`
- `combined_supergather_errors`

In [1]:
from pathlib import Path
import sqlite3
import json
import traceback
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from obspy import read, Stream, Trace, UTCDateTime

PROJECT_ROOT = Path("/Volumes/tachyon/LBSSP_DATA")
CATALOG_DB = PROJECT_ROOT / "catalog" / "lbssp_shot_catalog.sqlite"
CATALOG_DB.parent.mkdir(parents=True, exist_ok=True)
print("CATALOG_DB:", CATALOG_DB)

OUT_ROOT = PROJECT_ROOT / "combined_supergathers_v1"
OUT_ROOT.mkdir(parents=True, exist_ok=True)
COMPONENT = "Z"
ATTEMPT_OBSPY_GEODE_READ = True
print("OUT_ROOT:", OUT_ROOT)

CATALOG_DB: /Volumes/tachyon/LBSSP_DATA/catalog/lbssp_shot_catalog.sqlite
OUT_ROOT: /Volumes/tachyon/LBSSP_DATA/combined_supergathers_v1


## 1. Load inputs

In [2]:
REQUIRED = ["geode_nodal_common_shot_comparisons", "nodal_stack_files", "geode_events"]
OWNED = ["combined_supergathers", "combined_supergather_files", "combined_supergather_errors"]

with sqlite3.connect(CATALOG_DB) as conn:
    tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name", conn)["name"].tolist()
missing = [t for t in REQUIRED if t not in tables]
if missing:
    raise RuntimeError(f"Missing inputs: {missing}. Run 97 first.")

conn = sqlite3.connect(CATALOG_DB)
comparisons = pd.read_sql("SELECT * FROM geode_nodal_common_shot_comparisons", conn)
nodal_stack_files = pd.read_sql("SELECT * FROM nodal_stack_files", conn)
geode_events = pd.read_sql("SELECT * FROM geode_events", conn)
display(comparisons.head())

,comparison_id,stack_id,geode_event_id,survey,line,file_no,source_x_m,component,nodal_stack_mseed_path,nodal_stack_exists,geode_file_path,geode_file_exists,geode_read_format,geode_n_traces,status
0,CMP_NODALSTACK_T1_T1_1m_refraction_F3006_x0084.5m,NODALSTACK_T1_T1_1m_refraction_F3006_x0084.5m,GEODE_T1_1M_REFRACTION_F3006,T1_1m_refraction,T1,3006,84.5,Z,/Volumes/tachyon/LBSSP_DATA/nodal_stacked_by_g...,1,/Volumes/tachyon/LBSSP_DATA/GEODE_DATA/LBSSP_0...,1,SEG2,72.0,ready
1,CMP_NODALSTACK_T1_T1_1m_refraction_F3008_x0088.5m,NODALSTACK_T1_T1_1m_refraction_F3008_x0088.5m,GEODE_T1_1M_REFRACTION_F3008,T1_1m_refraction,T1,3008,88.5,Z,/Volumes/tachyon/LBSSP_DATA/nodal_stacked_by_g...,1,/Volumes/tachyon/LBSSP_DATA/GEODE_DATA/LBSSP_0...,1,SEG2,72.0,ready
2,CMP_NODALSTACK_T1_T1_1m_refraction_F3009_x0090.5m,NODALSTACK_T1_T1_1m_refraction_F3009_x0090.5m,GEODE_T1_1M_REFRACTION_F3009,T1_1m_refraction,T1,3009,90.5,Z,/Volumes/tachyon/LBSSP_DATA/nodal_stacked_by_g...,1,/Volumes/tachyon/LBSSP_DATA/GEODE_DATA/LBSSP_0...,1,SEG2,72.0,ready
3,CMP_NODALSTACK_T1_T1_1m_refraction_F3011_x0094.5m,NODALSTACK_T1_T1_1m_refraction_F3011_x0094.5m,GEODE_T1_1M_REFRACTION_F3011,T1_1m_refraction,T1,3011,94.5,Z,/Volumes/tachyon/LBSSP_DATA/nodal_stacked_by_g...,1,/Volumes/tachyon/LBSSP_DATA/GEODE_DATA/LBSSP_0...,1,SEG2,72.0,ready
4,CMP_NODALSTACK_T1_T1_1m_refraction_F3012_x0096.5m,NODALSTACK_T1_T1_1m_refraction_F3012_x0096.5m,GEODE_T1_1M_REFRACTION_F3012,T1_1m_refraction,T1,3012,96.5,Z,/Volumes/tachyon/LBSSP_DATA/nodal_stacked_by_g...,1,/Volumes/tachyon/LBSSP_DATA/GEODE_DATA/LBSSP_0...,1,SEG2,72.0,ready


## 2. Build supergather manifests and optional Geode conversions

In [3]:
super_rows = []
file_rows = []
error_rows = []

for _, row in comparisons.iterrows():
    super_id = f"SUPER_{row['stack_id']}"
    out_dir = OUT_ROOT / str(row["line"]) / str(row["survey"]) / super_id
    out_dir.mkdir(parents=True, exist_ok=True)

    nodal_path = row.get("nodal_stack_mseed_path")
    geode_path = row.get("geode_file_path")

    nodal_exists = Path(str(nodal_path)).exists() if nodal_path not in [None, "None", "nan"] else False
    geode_exists = Path(str(geode_path)).exists() if geode_path not in [None, "None", "nan"] else False

    n_nodal_traces = None
    n_geode_traces = None

    try:
        if nodal_exists:
            st_nodal = read(str(nodal_path))
            n_nodal_traces = len(st_nodal)
            pointer = out_dir / "nodal_stack_mseed_path.txt"
            pointer.write_text(str(nodal_path))
            file_rows.append({"supergather_id": super_id, "component": COMPONENT, "file_type": "nodal_mseed_pointer", "file_path": str(pointer)})

        if ATTEMPT_OBSPY_GEODE_READ and geode_exists:
            try:
                st_geode = read(str(geode_path))
                n_geode_traces = len(st_geode)
                out_geode_mseed = out_dir / f"{super_id}_GEODE_raw.mseed"
                st_geode.write(str(out_geode_mseed), format="MSEED")
                file_rows.append({"supergather_id": super_id, "component": COMPONENT, "file_type": "geode_mseed_converted", "file_path": str(out_geode_mseed)})
            except Exception as e:
                error_rows.append({"supergather_id": super_id, "stage": "read_geode", "file_path": str(geode_path), "error": repr(e), "traceback": traceback.format_exc()})

        manifest = {
            "supergather_id": super_id,
            "stack_id": row["stack_id"],
            "geode_event_id": row["geode_event_id"],
            "survey": row["survey"],
            "line": row["line"],
            "file_no": row["file_no"],
            "source_x_m": row["source_x_m"],
            "nodal_stack_mseed_path": nodal_path,
            "geode_file_path": geode_path,
            "nodal_exists": bool(nodal_exists),
            "geode_exists": bool(geode_exists),
        }
        manifest_path = out_dir / "manifest.json"
        manifest_path.write_text(json.dumps(manifest, indent=2, default=str))
        file_rows.append({"supergather_id": super_id, "component": COMPONENT, "file_type": "manifest_json", "file_path": str(manifest_path)})

        super_rows.append({
            "supergather_id": super_id,
            "stack_id": row["stack_id"],
            "geode_event_id": row["geode_event_id"],
            "survey": row["survey"],
            "line": row["line"],
            "file_no": row["file_no"],
            "source_x_m": row["source_x_m"],
            "component": COMPONENT,
            "nodal_stack_mseed_path": nodal_path,
            "geode_file_path": geode_path,
            "nodal_exists": bool(nodal_exists),
            "geode_exists": bool(geode_exists),
            "n_nodal_traces": n_nodal_traces,
            "n_geode_traces": n_geode_traces,
            "output_dir": str(out_dir),
            "status": "indexed",
        })
    except Exception as e:
        error_rows.append({"supergather_id": super_id, "stage": "build_supergather", "error": repr(e), "traceback": traceback.format_exc()})

combined_supergathers = pd.DataFrame(super_rows)
combined_supergather_files = pd.DataFrame(file_rows)
combined_supergather_errors = pd.DataFrame(error_rows)
display(combined_supergathers.head())
display(combined_supergather_errors.head())

/opt/anaconda3/envs/flovopy_plus/lib/python3.12/site-packages/obspy/io/seg2/seg2.py:365: UserWarning: Many companies use custom defined SEG2 header variables. This might cause basic header information reflected in the single traces' stats to be wrong (e.g. recording delays, first sample number, station code names, ..). Please check the complete list of additional unmapped header fields that gets stored in Trace.stats.seg2 and/or the manual of the source of the SEG2 files for fields that might influence e.g. trace start times.
  warnings.warn(WARNING_HEADER)


,supergather_id,stack_id,geode_event_id,survey,line,file_no,source_x_m,component,nodal_stack_mseed_path,geode_file_path,nodal_exists,geode_exists,n_nodal_traces,n_geode_traces,output_dir,status
0,SUPER_NODALSTACK_T1_T1_1m_refraction_F3006_x00...,NODALSTACK_T1_T1_1m_refraction_F3006_x0084.5m,GEODE_T1_1M_REFRACTION_F3006,T1_1m_refraction,T1,3006,84.5,Z,/Volumes/tachyon/LBSSP_DATA/nodal_stacked_by_g...,/Volumes/tachyon/LBSSP_DATA/GEODE_DATA/LBSSP_0...,True,True,34,72.0,/Volumes/tachyon/LBSSP_DATA/combined_supergath...,indexed
1,SUPER_NODALSTACK_T1_T1_1m_refraction_F3008_x00...,NODALSTACK_T1_T1_1m_refraction_F3008_x0088.5m,GEODE_T1_1M_REFRACTION_F3008,T1_1m_refraction,T1,3008,88.5,Z,/Volumes/tachyon/LBSSP_DATA/nodal_stacked_by_g...,/Volumes/tachyon/LBSSP_DATA/GEODE_DATA/LBSSP_0...,True,True,34,72.0,/Volumes/tachyon/LBSSP_DATA/combined_supergath...,indexed
2,SUPER_NODALSTACK_T1_T1_1m_refraction_F3009_x00...,NODALSTACK_T1_T1_1m_refraction_F3009_x0090.5m,GEODE_T1_1M_REFRACTION_F3009,T1_1m_refraction,T1,3009,90.5,Z,/Volumes/tachyon/LBSSP_DATA/nodal_stacked_by_g...,/Volumes/tachyon/LBSSP_DATA/GEODE_DATA/LBSSP_0...,True,True,34,72.0,/Volumes/tachyon/LBSSP_DATA/combined_supergath...,indexed
3,SUPER_NODALSTACK_T1_T1_1m_refraction_F3011_x00...,NODALSTACK_T1_T1_1m_refraction_F3011_x0094.5m,GEODE_T1_1M_REFRACTION_F3011,T1_1m_refraction,T1,3011,94.5,Z,/Volumes/tachyon/LBSSP_DATA/nodal_stacked_by_g...,/Volumes/tachyon/LBSSP_DATA/GEODE_DATA/LBSSP_0...,True,True,34,72.0,/Volumes/tachyon/LBSSP_DATA/combined_supergath...,indexed
4,SUPER_NODALSTACK_T1_T1_1m_refraction_F3012_x00...,NODALSTACK_T1_T1_1m_refraction_F3012_x0096.5m,GEODE_T1_1M_REFRACTION_F3012,T1_1m_refraction,T1,3012,96.5,Z,/Volumes/tachyon/LBSSP_DATA/nodal_stacked_by_g...,/Volumes/tachyon/LBSSP_DATA/GEODE_DATA/LBSSP_0...,True,True,34,72.0,/Volumes/tachyon/LBSSP_DATA/combined_supergath...,indexed


""


## 3. Safe database write

In [4]:
if combined_supergather_files is None or len(combined_supergather_files.columns) == 0:
    combined_supergather_files = pd.DataFrame(columns=["supergather_id", "component", "file_type", "file_path"])
if combined_supergather_errors is None or len(combined_supergather_errors.columns) == 0:
    combined_supergather_errors = pd.DataFrame(columns=["supergather_id", "stage", "error"])
if combined_supergathers is None or len(combined_supergathers.columns) == 0:
    combined_supergathers = pd.DataFrame(columns=["supergather_id", "stack_id", "status"])

with sqlite3.connect(CATALOG_DB) as conn:
    for tname in OWNED:
        conn.execute(f'DROP TABLE IF EXISTS "{tname}"')
    combined_supergathers.to_sql("combined_supergathers", conn, if_exists="fail", index=False)
    combined_supergather_files.to_sql("combined_supergather_files", conn, if_exists="fail", index=False)
    combined_supergather_errors.to_sql("combined_supergather_errors", conn, if_exists="fail", index=False)
    conn.commit()

combined_supergathers.to_csv(OUT_ROOT / "combined_supergathers.csv", index=False)
combined_supergather_files.to_csv(OUT_ROOT / "combined_supergather_files.csv", index=False)
combined_supergather_errors.to_csv(OUT_ROOT / "combined_supergather_errors.csv", index=False)

print("Wrote 98-owned tables:", OWNED)
display(combined_supergathers.groupby(["survey", "status"]).size().reset_index(name="n") if len(combined_supergathers) else combined_supergathers)

Wrote 98-owned tables: ['combined_supergathers', 'combined_supergather_files', 'combined_supergather_errors']


,survey,status,n
0,T1_1m_refraction,indexed,34
1,T1_2m_refraction,indexed,23
2,T1_streamer_masw,indexed,66
3,T3_1m_refraction,indexed,27
